# regime 2 / Compare Results — LoRA vs QLoRA vs two-factor vs three-factor
Loads `results/{lora,qlora,two_factor,three_factor}.json` from Drive (run those first; missing ones are skipped).

## 1. Setup + Load Results

In [ ]:
import os, json, math, torch
import matplotlib.pyplot as plt
USE_DRIVE, DRIVE_SUBDIR = True, 'Section8_regime2'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:', e); STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results')
METHODS = ['lora', 'qlora', 'two_factor', 'three_factor']
LABELS  = {'lora':'LoRA (backprop)', 'qlora':'QLoRA (4-bit)', 'two_factor':'Two-factor Hebbian', 'three_factor':'Three-factor (this work)'}
COLORS  = {'lora':'#1F3864', 'qlora':'#2E7D32', 'two_factor':'#B8860B', 'three_factor':'#C62828'}
R = {}
for m in METHODS:
    p = os.path.join(RESULTS_DIR, f'{m}.json')
    if os.path.exists(p):
        R[m] = json.load(open(p)); s = R[m]['summary']; md = R[m]['meta']
        print(f'loaded {m:14} {md["total_steps"]:>8,} steps  {md["wall_clock_sec"]/3600:5.2f}h  final acc {s["final_acc"]:.3f}  best loss {s["best_loss"]:.4f}')
    else:
        print(f'MISSING {p} (run the {m} notebook first)')

## 2. Configuration

In [ ]:
for m in R:
    md=R[m]['meta']; print('='*60); print(LABELS[m]); print('  P_train=%d epochs=%.3f'%(md.get('P_trainable',0),md['epochs'])); print('  config:', json.dumps(md['config']))

## 3. Compare Loss Curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_acc'],  'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test loss'); ax[0].set_title('CIFAR-10 test loss vs time'); ax[0].legend()
ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('CIFAR-10 test accuracy vs time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Compare Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m == 'three_factor': return st * 2 * R[m]['meta']['config'].get('M', 0)  # loop probes
    if m in ('lora', 'qlora'): return st * 3                                    # fwd + ~2x bwd
    return st * 2                                                               # two-factor
print(f'{"method":26}{"P_train":>10}{"steps":>9}{"fwd-equiv":>13}{"wall h":>8}{"peak MB":>9}')
print('-'*75)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:26}{md.get("P_trainable",0):>10,}{md["total_steps"]:>9,}{fwd_equiv(m):>13,}'
          f'{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## Peak training memory — backprop-LoRA vs. forward-only (frozen base, no tape)

regime 2 freezes ViT-Base (86M) and trains only LoRA adapters. Backprop must still push gradients **through** the frozen base, so it retains the base's activation tape; the forward-only rule only forward-passes it. HF-dependent — runs on the Colab runtime (`transformers`+`peft`); guarded so it never crashes. Left unexecuted.

In [ ]:
# ============================================================================
# Peak training memory: backprop-LoRA vs the forward-only three-factor rule.
# regime 2 = FROZEN ViT-Base (86M) + small LoRA adapters. To update the adapters,
# backprop must push gradients THROUGH the frozen base, so it still retains the
# base's activation tape; the forward-only rule only forward-passes the base
# (each probe = a forward eval) and stores none of it -> training memory stays at
# the base's inference footprint. HF-dependent: runs on the Colab runtime
# (transformers + peft). Measured exactly via saved_tensors_hooks; fully guarded.
# Writes peak_training_memory.csv into exports/ (rides along in the results zip).
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
_MEM_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
_MEM_BATCH  = 8    # keep the diagnostic light; activation tape ~ linear in batch
PEAK_MEM_ROWS = []
try:
    from transformers import ViTForImageClassification
    from peft import LoraConfig, get_peft_model
    import re as _re
    _MODEL = globals().get('MODEL_NAME', 'google/vit-base-patch16-224')
    _NC = int(globals().get('NUM_CLASSES', 10)); _IMG = int(globals().get('IMG_SIZE', 224))
    _R = int(globals().get('LORA_R', 4)); _A = int(globals().get('LORA_ALPHA', 8))
    _TGTS = list(globals().get('LORA_TARGETS', ['query', 'value']))
    _K = int(globals().get('LORA_LAST_K_LAYERS', 2))
    _base = ViTForImageClassification.from_pretrained(_MODEL, num_labels=_NC,
                                                      ignore_mismatched_sizes=True).to(_MEM_DEVICE)
    _lins = [(n, m) for n, m in _base.named_modules() if isinstance(m, nn.Linear)]
    def _lidx(n):
        mm = _re.search(r'(?:^|\.)(?:layers|layer|blocks|block|h)\.(\d+)(?:\.|$)', n)
        return int(mm.group(1)) if mm else None
    _maxL = max((_lidx(n) for n, _ in _lins if _lidx(n) is not None), default=-1)
    _keep = set(range(_maxL + 1 - _K, _maxL + 1)) if _maxL >= 0 else set()
    _t = sorted({n for n, _ in _lins if _lidx(n) in _keep and any(t in n.split('.')[-1] for t in _TGTS)})
    _model = get_peft_model(_base, LoraConfig(r=_R, lora_alpha=_A, lora_dropout=0.0,
                            target_modules=_t, modules_to_save=['classifier'], bias='none')).to(_MEM_DEVICE)
    _tot = sum(p.numel() for p in _model.parameters())
    _tr  = sum(p.numel() for p in _model.parameters() if p.requires_grad)
    _pp = {p.untyped_storage().data_ptr() for p in _model.parameters()}
    _seen, _isp = {}, {}
    def _pack(t):
        st = t.untyped_storage(); dp = st.data_ptr(); _seen[dp] = st.nbytes(); _isp[dp] = (dp in _pp); return t
    def _unpack(t): return t
    _x = torch.randn(_MEM_BATCH, 3, _IMG, _IMG, device=_MEM_DEVICE)
    _y = torch.randint(0, _NC, (_MEM_BATCH,), device=_MEM_DEVICE)
    with torch.autograd.graph.saved_tensors_hooks(_pack, _unpack):
        _out = _model(pixel_values=_x).logits; F.cross_entropy(_out, _y).backward()
    _tape = sum(nb for dp, nb in _seen.items() if not _isp[dp]); MB = 1e6
    _tb = int(globals().get('BATCH', 64)); _scale = _tb / _MEM_BATCH
    print("=" * 80)
    print("PEAK TRAINING MEMORY  |  S8 regime 2 - LoRA fine-tune (frozen ViT-Base, 86M)")
    print(f"  total params {_tot:,} ({_tot*4/MB:.0f} MB)  |  trainable LoRA+head {_tr:,} ({_tr*4/MB:.2f} MB)")
    print(f"  measured at batch {_MEM_BATCH}  (tape ~ linear -> x{_scale:.0f} at training batch {_tb}):")
    print(f"    tape backprop-LoRA retains through the frozen base : {_tape/MB:9.1f} MB  (~{_tape/MB*_scale:.0f} MB @ batch {_tb})")
    print(f"    forward-only three-factor rule                     : {0.0:9.1f} MB  (no tape -- forward passes only)")
    print("  -> frozen weights do NOT spare backprop the tape; forward-only stays at inference footprint.")
    print("=" * 80)
    PEAK_MEM_ROWS = [{"section": "S8 regime 2 - LoRA (frozen ViT-Base 86M)", "arch": "ViT-Base+LoRA",
                      "total_params": _tot, "trainable_params": _tr, "meas_batch": _MEM_BATCH,
                      "train_batch": _tb, "activation_tape_MB_meas": round(_tape/MB, 1),
                      "activation_tape_MB_at_train_batch": round(_tape/MB*_scale, 1),
                      "forward_only_tape_MB": 0.0}]
except Exception as _e:
    print("[Peak-memory cell] Live measurement skipped:", type(_e).__name__, str(_e)[:150])
    print("Needs the Colab runtime with transformers + peft and the base model available.")
    print("What it shows: backprop-LoRA must retain the FROZEN ViT-Base activation tape to push")
    print("gradients back to the adapters, while the forward-only three-factor rule only forward-")
    print("passes the base (each probe = a forward eval) and stores NO tape -> its training memory")
    print("equals the base's inference footprint. Re-run on the A100 runtime for the measured MB.")

# --- persist so the results zip / Drive picks it up (rides next to summary.csv) ---
try:
    import os as _os, csv as _csv
    if PEAK_MEM_ROWS and 'STORE' in globals():
        _ed = _os.path.join(STORE, 'exports'); _os.makedirs(_ed, exist_ok=True)
        with open(_os.path.join(_ed, 'peak_training_memory.csv'), 'w', newline='') as _f:
            _w = _csv.DictWriter(_f, fieldnames=list(PEAK_MEM_ROWS[0].keys())); _w.writeheader()
            for _r in PEAK_MEM_ROWS: _w.writerow(_r)
        print('  saved ->', _os.path.join(_ed, 'peak_training_memory.csv'))
except Exception as _e:
    print('  (peak_training_memory.csv not written:', _e, ')')


## 5. Summary table

In [ ]:
print(f'{"Experiment":26}{"Epochs":>9}{"First loss":>12}{"Final loss":>12}{"Best loss":>11}{"Reduc %":>9}{"Test acc":>10}')
print('-'*89)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:26}{md["epochs"]:>9.3f}{s["initial_loss"]:>12.4f}{s["final_loss"]:>12.4f}'
          f'{s["best_loss"]:>11.4f}{s["reduction_pct"]:>9.1f}{s["final_acc"]:>10.3f}')
print('\n(Equal wall-clock: LoRA/QLoRA get thousands of steps; the forward-only three-factor rule gets a few '
      'hundred because each probe is a full ViT-Base forward -- the honest cost of backprop-free big-model adaptation.)')

## Export figures + CSVs → Drive (and download a zip)

Writes `curves.csv`, `summary.csv`, the comparison figure, and the `peak_training_memory.csv` from the cell above into `STORE/exports/`, then zips and downloads it. Left unexecuted.

In [ ]:
# -- Export comparison: figures (PNG) + data (CSV) -> a Drive folder, then download a zip --
# Schema-robust (dynamic key union). Also picks up peak_training_memory.csv written by the
# "Peak training memory" cell above, so it rides along in the zip next to summary.csv.
import os, csv, glob, shutil
import matplotlib.pyplot as plt
EXPORT_DIR = os.path.join(STORE, 'exports'); os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) CSVs -- long-format curves + one summary row per method (robust to differing keys across methods)
if R:
    ckeys = []
    for m in R:
        for k in R[m]['curve']:
            if k not in ckeys: ckeys.append(k)
    with open(os.path.join(EXPORT_DIR, 'curves.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method'] + ckeys)
        for m in R:
            c = R[m]['curve']; n = max((len(c[k]) for k in c), default=0)
            for i in range(n):
                w.writerow([m] + [c[k][i] if (k in c and i < len(c[k])) else '' for k in ckeys])
    skeys = []
    for m in R:
        for k in R[m]['summary']:
            if k not in skeys: skeys.append(k)
    with open(os.path.join(EXPORT_DIR, 'summary.csv'), 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['method', 'P', 'total_steps', 'wall_clock_h', 'M', 'cos'] + skeys)
        for m in R:
            md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0
            cos = round((M/(M+md['P']+1))**0.5, 4) if M else ''
            w.writerow([m, md['P'], md.get('total_steps', ''), round(md['wall_clock_sec']/3600, 3), M, cos]
                       + [s.get(k, '') for k in skeys])
    print('wrote curves.csv, summary.csv')

# 2) Figure -- test loss & accuracy vs wall-clock (guarded to the keys present)
def _save(fig, name): fig.savefig(os.path.join(EXPORT_DIR, name), dpi=120, bbox_inches='tight'); plt.close(fig)
if R and all('t_sec' in R[m]['curve'] for m in R):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for m in R:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        if 'test_loss' in c: ax[0].plot(hrs, c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
        if 'test_acc'  in c: ax[1].plot(hrs, c['test_acc'],  'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_xlabel('hours'); ax[0].set_ylabel('test loss'); ax[0].set_title('Test loss vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('Test accuracy vs time'); ax[1].legend()
    _save(fig, 'curves_all_methods.png'); print('wrote curves_all_methods.png')

# 3) Copy any per-run progress PNGs the trainers saved to Drive
for d in {globals().get('RESULTS_DIR', STORE), globals().get('CKPT_DIR', STORE), STORE, os.path.join(STORE, 'figures')}:
    for png in glob.glob(os.path.join(d, '*.png')):
        try: shutil.copy(png, EXPORT_DIR)
        except Exception: pass

print('\nexport folder:', EXPORT_DIR); print(' ', sorted(os.listdir(EXPORT_DIR)))

# 4) Zip the folder (kept on Drive too) and auto-download it in Colab
_ztarget = '/content' if os.path.isdir('/content') else os.path.dirname(EXPORT_DIR)
_zip = shutil.make_archive(os.path.join(_ztarget, f'{DRIVE_SUBDIR}_exports'), 'zip', EXPORT_DIR)
try:
    if os.path.abspath(os.path.dirname(_zip)) != os.path.abspath(STORE): shutil.copy(_zip, STORE)
except Exception as e: print('(could not copy zip to Drive):', e)
print('zip:', _zip)
try:
    from google.colab import files; files.download(_zip)
except Exception as e:
    print('(download only runs in Colab):', e)
